# AI Gaming Editor - Colab Demo\n\nتجريب مشروع تحليل فيديو الـ Gaming وتحويله لريلز احترافية\n\n**المتطلبات:**\n- حساب Google Drive فيه مجلد `videos` يحتوي على فيديو gaming\n- اختيار GPU Runtime من Colab (T4 مثلاً)\n\n**الخطوات:**\n1. الاتصال بـ Google Drive\n2. تثبيت المكتبات المطلوبة\n3. استنساخ المشروع\n4. اختيار فيديو من Drive\n5. تشغيل Pipeline التحليل\n6. عرض النتائج وتحميل الريل

## 1. الاتصال بـ Google Drive

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

## 2. التحقق من GPU

In [ ]:
!nvidia-smi\nimport torch\nprint(f"\\nPyTorch CUDA: {torch.cuda.is_available()}")\nif torch.cuda.is_available():\n    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. تثبيت المكتبات المطلوبة

In [ ]:
!pip install -q opencv-python-headless numpy easyocr librosa scenedetect[opencv] ultralytics openai\n!pip install -q git+https://github.com/m-bain/whisperx.git

## 4. استنساخ المشروع

In [ ]:
import os\n\n%cd /content\n\nif os.path.exists("ai_gaming_editor"):\n    !rm -rf ai_gaming_editor\n\n!git clone https://github.com/sadek-marouf/ai_gaming_editor.git\n\n%cd ai_gaming_editor\n\nprint("\\nProject files:")\n!ls -la

## 5. اختيار فيديو من Google Drive\n\nتأكد أن عندك مجلد `videos` في الـ Drive يحتوي على فيديو gaming.\nالخلية التالية تعرض الفيديوهات المتوفرة وتخليك تختار واحد.

In [ ]:
import os\nimport shutil\n\nDRIVE_VIDEOS_DIR = "/content/drive/MyDrive/videos"\nLOCAL_VIDEO_DIR = "/content/ai_gaming_editor/input_videos"\n\nos.makedirs(LOCAL_VIDEO_DIR, exist_ok=True)\n\nvideo_extensions = ('.mp4', '.avi', '.mkv', '.mov', '.webm', '.flv')\n\nif not os.path.exists(DRIVE_VIDEOS_DIR):\n    print(f"ERROR: '{DRIVE_VIDEOS_DIR}' not found!")\n    print("Please create a 'videos' folder in your Google Drive root and add gaming videos to it.")\nelse:\n    videos = [\n        f for f in os.listdir(DRIVE_VIDEOS_DIR)\n        if f.lower().endswith(video_extensions)\n    ]\n\n    if not videos:\n        print(f"No video files found in {DRIVE_VIDEOS_DIR}")\n        print(f"Supported formats: {video_extensions}")\n    else:\n        print(f"Found {len(videos)} video(s) in Drive/videos:\\n")\n        for i, v in enumerate(videos):\n            size_mb = os.path.getsize(os.path.join(DRIVE_VIDEOS_DIR, v)) / (1024 * 1024)\n            print(f"  [{i}] {v} ({size_mb:.1f} MB)")

## 6. نسخ الفيديو المختار\n\nغيّر `VIDEO_INDEX` لاختيار فيديو مختلف من القائمة أعلاه.

In [ ]:
# ==============================\n# اختر رقم الفيديو من القائمة\n# ==============================\nVIDEO_INDEX = 0\n\nsrc = os.path.join(DRIVE_VIDEOS_DIR, videos[VIDEO_INDEX])\ndst = os.path.join(LOCAL_VIDEO_DIR, videos[VIDEO_INDEX])\n\nprint(f"Copying: {videos[VIDEO_INDEX]}...")\nshutil.copy2(src, dst)\n\nVIDEO_PATH = dst\nprint(f"Video ready: {VIDEO_PATH}")\nprint(f"Size: {os.path.getsize(VIDEO_PATH) / (1024*1024):.1f} MB")

## 7. معلومات الفيديو

In [ ]:
import cv2\n\ncap = cv2.VideoCapture(VIDEO_PATH)\nfps = cap.get(cv2.CAP_PROP_FPS)\ntotal_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\nwidth = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\nheight = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\nduration = total_frames / fps if fps > 0 else 0\ncap.release()\n\nprint(f"Resolution: {width}x{height}")\nprint(f"FPS: {fps}")\nprint(f"Total Frames: {total_frames}")\nprint(f"Duration: {duration:.1f} sec ({duration/60:.1f} min)")

## 8. إعداد Groq AI (اختياري)\n\nإذا عندك مفتاح Groq API، فعّله هنا للحصول على تقييم ذكاء اصطناعي للمقاطع.\nإذا ما عندك، المشروع يشتغل بدونه بشكل طبيعي.

In [ ]:
# ==============================\n# اختياري: ضع مفتاح Groq API\n# اتركه فارغ إذا ما عندك\n# ==============================\nimport os\n\nGROQ_API_KEY = ""  # ضع المفتاح هنا إذا عندك\n\nif GROQ_API_KEY:\n    os.environ["GROQ_API_KEY"] = GROQ_API_KEY\n    print("Groq AI ENABLED")\nelse:\n    print("Groq AI DISABLED (optional - project works without it)")

## 9. تشغيل Pipeline التحليل وتوليد الريل\n\nهذه الخلية تشغل كامل Pipeline:\n1. استخراج الصوت\n2. تفريغ النص (WhisperX)\n3. تحليل متوازي (صوت، حركة، مشاهد بصرية، وجوه، مشاهد)\n4. كشف لحظات الـ Gaming الحماسية\n5. تقييم وترتيب المقاطع\n6. توليد الريل النهائي

In [ ]:
import sys\nsys.path.insert(0, "/content/ai_gaming_editor")\n\nfrom core.pipeline import Pipeline\n\n# ==============================\n# اعدادات التشغيل\n# ==============================\nQUALITY = "medium"   # low / medium / high\nWORKERS = 4          # عدد العمليات المتوازية\nOUTPUT_DIR = "/content/ai_gaming_editor/outputs"\n\npipeline = Pipeline(\n    VIDEO_PATH,\n    output_dir=OUTPUT_DIR,\n    quality=QUALITY,\n    parallel_workers=WORKERS,\n)\n\nprint("Starting pipeline...\\n")\nresult = pipeline.run()\n\nif result:\n    print(f"\\n{'='*60}")\n    print(f"SUCCESS! Reel generated: {result}")\n    print(f"{'='*60}")\nelse:\n    print(f"\\n{'='*60}")\n    print(f"FAILED: Could not generate reel")\n    print(f"{'='*60}")

## 10. عرض نتائج التحليل

In [ ]:
import json\nimport glob\n\n# Find segments.json\nbase_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]\nsegments_path = os.path.join(OUTPUT_DIR, base_name, "segments.json")\n\nif os.path.exists(segments_path):\n    with open(segments_path, "r", encoding="utf-8") as f:\n        segments = json.load(f)\n\n    print(f"Top {len(segments)} segments selected for reel:\\n")\n    print(f"{'#':<4} {'Start':>8} {'End':>8} {'Score':>8} {'Audio':>8} {'Motion':>8} {'Hype':>8} Text")\n    print("-" * 90)\n\n    for i, seg in enumerate(segments, 1):\n        text_preview = seg['text'].replace('\\\\N', ' ')[:40]\n        print(\n            f"{i:<4} {seg['start']:>8.1f} {seg['end']:>8.1f} \"\n            f\"{seg['score']:>8.3f} {seg.get('audio', 0):>8.3f} \"\n            f\"{seg.get('motion', 0):>8.3f} {seg.get('hype', 0):>8.3f} \"\n            f\"{text_preview}\"\n        )\nelse:\n    print("segments.json not found - pipeline may have failed")

## 11. عرض الريل في Colab

In [ ]:
from IPython.display import HTML\nfrom base64 import b64encode\n\nif result and os.path.exists(result):\n    reel_size = os.path.getsize(result) / (1024 * 1024)\n    print(f"Reel size: {reel_size:.1f} MB")\n\n    if reel_size < 50:\n        with open(result, \"rb\") as f:\n            video_data = f.read()\n        video_b64 = b64encode(video_data).decode()\n        display(HTML(f\"\"\"\n        <video width=\"360\" height=\"640\" controls>\n            <source src=\"data:video/mp4;base64,{video_b64}\" type=\"video/mp4\">\n        </video>\n        \"\"\"))\n    else:\n        print(f\"Video too large to display inline ({reel_size:.0f} MB)\")\n        print(f\"Download it from: {result}\")\nelse:\n    print(\"No reel to display\")

## 12. حفظ الريل على Google Drive

In [ ]:
DRIVE_OUTPUT = "/content/drive/MyDrive/videos/reels"\nos.makedirs(DRIVE_OUTPUT, exist_ok=True)\n\nif result and os.path.exists(result):\n    output_name = f"{base_name}_reel.mp4"\n    drive_path = os.path.join(DRIVE_OUTPUT, output_name)\n    shutil.copy2(result, drive_path)\n    print(f"Reel saved to Drive: {drive_path}")\n\n    # also copy segments analysis\n    if os.path.exists(segments_path):\n        analysis_dest = os.path.join(DRIVE_OUTPUT, f"{base_name}_segments.json")\n        shutil.copy2(segments_path, analysis_dest)\n        print(f"Analysis saved to Drive: {analysis_dest}")\nelse:\n    print("No reel to save")

## 13. تنظيف الملفات المؤقتة (اختياري)

In [ ]:
# uncomment to clean up\n# !rm -rf /content/ai_gaming_editor/outputs\n# !rm -rf /content/ai_gaming_editor/input_videos\n\nprint("Done! Check your Drive/videos/reels folder for the output.")